In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Assignment16").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 04:48:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/17 04:48:43 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
print("Spark Version:", spark.version)

Spark Version: 3.5.6


In [4]:
rdd = spark.sparkContext.textFile("employee.csv")


In [5]:
data = rdd.collect()
for i in data:
    print(i)

id,name,department,salary
1,Amit,IT,55000
2,Rahul,HR,40000
3,Neha,IT,65000
4,Priya,Finance,70000
5,Karan,IT,50000
6,Simran,HR,45000
7,Rohit,Finance,60000


In [6]:

header = rdd.first()
employees = rdd.filter(lambda x: x != header)
employees.collect()

['1,Amit,IT,55000',
 '2,Rahul,HR,40000',
 '3,Neha,IT,65000',
 '4,Priya,Finance,70000',
 '5,Karan,IT,50000',
 '6,Simran,HR,45000',
 '7,Rohit,Finance,60000']

In [7]:
employees = employees.map(
    lambda x: x.split(",") )
employees.collect()

[['1', 'Amit', 'IT', '55000'],
 ['2', 'Rahul', 'HR', '40000'],
 ['3', 'Neha', 'IT', '65000'],
 ['4', 'Priya', 'Finance', '70000'],
 ['5', 'Karan', 'IT', '50000'],
 ['6', 'Simran', 'HR', '45000'],
 ['7', 'Rohit', 'Finance', '60000']]

In [9]:
# sort all employees by salary in descending order
sort_emp = employees.sortBy(
     lambda x: int(x[3]),
    ascending=False )

sort_emp.collect()



[['4', 'Priya', 'Finance', '70000'],
 ['3', 'Neha', 'IT', '65000'],
 ['7', 'Rohit', 'Finance', '60000'],
 ['1', 'Amit', 'IT', '55000'],
 ['5', 'Karan', 'IT', '50000'],
 ['6', 'Simran', 'HR', '45000'],
 ['2', 'Rahul', 'HR', '40000']]

In [10]:
for row in sort_emp.collect():
    print(row)

['4', 'Priya', 'Finance', '70000']
['3', 'Neha', 'IT', '65000']
['7', 'Rohit', 'Finance', '60000']
['1', 'Amit', 'IT', '55000']
['5', 'Karan', 'IT', '50000']
['6', 'Simran', 'HR', '45000']
['2', 'Rahul', 'HR', '40000']


In [12]:
# Total Salary Department Wise

dept_sal =  employees.map(
    lambda x: (x[2], int(x[3]))  )

dept_total = dept_sal.reduceByKey(
    lambda a, b: a + b )
dept_total.collect()


[('IT', 170000), ('HR', 85000), ('Finance', 130000)]

In [13]:
for dept, total in dept_total.collect():
    print(dept, total)

IT 170000
HR 85000
Finance 130000


In [14]:
# Top 3 Highest Paid Employees
top3 = sort_emp.take(3)

for emp in top3:
    print(emp)

['4', 'Priya', 'Finance', '70000']
['3', 'Neha', 'IT', '65000']
['7', 'Rohit', 'Finance', '60000']


In [16]:
# Savetop3_rdd Top 3 to Output File
top3_rdd = spark.sparkContext.parallelize(top3)
top3_rdd.saveAsTextFile("Output/top3_employees")